# 12 -- Optymalizacja wielokryterialna (GA + Pareto)

W nb 10/11 widzieliśmy, że **różne kryteria** (max NPV, min LCOE, min payback) wskazują
na różne punkty projektowe. To naturalna konsekwencja **konfliktów celów** w projektowaniu:
duża inwestycja → wysokie NPV, ale wyższy LCOE; mała inwestycja → niski LCOE i payback,
ale mniejsze NPV.

**Optymalizacja wielokryterialna** nie szuka *jednego* optimum, lecz **frontu Pareto** —
zbioru rozwiązań, w których **nie da się poprawić jednego celu bez pogorszenia drugiego**.

### Dlaczego algorytm genetyczny (GA)

- **Mieszane zmienne**: continuous (install_pct, D_pipe) + discrete (n_turbines, typ turbiny)
- **Funkcja celu jest kosztowna** (każda ewaluacja = pełny `compute_power`)
- **Wiele lokalnych ekstremów** — GA dobrze sobie z tym radzi
- **Dobry suport dla multi-objective** — algorytm NSGA-II zwraca front Pareto bezpośrednio

Używamy biblioteki **`pymoo`** (jeśli brak: `pip install pymoo`).

### Decyzje projektowe (zmienne decyzyjne)

| Zmienna | Typ | Zakres |
|---------|-----|--------|
| `install_pct` | continuous | 10–60% przekroczenia |
| `n_turbines` | discrete | 1, 2, 3, 4 |
| `turbine_key` | categorical | kaplan, propeller, crossflow |

### Cele optymalizacji

| Cel | Kierunek | Interpretacja |
|-----|----------|---------------|
| **NPV** | maksymalizujemy | całkowity zysk |
| **LCOE** | minimalizujemy | koszt jednej MWh |

### Ograniczenia

- $H_{design}$ musi mieścić się w zakresie wybranego typu turbiny
- Plant musi pracować co najmniej 100 dni/rok

---

## Konfiguracja

In [1]:
import sys
sys.path.insert(0, '..')

import warnings
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# pymoo dla optymalizacji genetycznej
try:
    from pymoo.core.problem import ElementwiseProblem
    from pymoo.algorithms.moo.nsga2 import NSGA2
    from pymoo.operators.crossover.sbx import SBX
    from pymoo.operators.mutation.pm import PM
    from pymoo.operators.sampling.rnd import FloatRandomSampling
    from pymoo.optimize import minimize
    PYMOO_OK = True
except ImportError:
    print('UWAGA: brak pymoo. Zainstaluj: pip install pymoo')
    PYMOO_OK = False

from src.imgw_data import load_processed
from src.hydrology import (
    average_sorted_year, interpolate_q_to_location, environmental_flow,
)
from src.watershed import get_station_area, find_gauge, get_catchment_area
from src.losses import trash_rack_loss, minor_loss
from src.turbine import TURBINE_CATALOG
from src.production import compute_power, annual_energy, capacity_factor, operating_hours
from src.costs import total_investment, economic_analysis, EUR_PLN_RATE

pd.set_option('display.max_columns', 15)
print('Moduly zaladowane.')

Moduly zaladowane.


## Krok 1: Dane wejsciowe (jak w nb 10/11)

In [2]:
df = load_processed('../data/processed/daily_hydro_clean.parquet')
STATION_UP = '151160170'
STATION_DOWN = '151160150'
A_UP = get_station_area(STATION_UP)
A_DOWN = get_station_area(STATION_DOWN)
g_up = find_gauge(STATION_UP); g_down = find_gauge(STATION_DOWN)
lat_ew = (g_up['lat'] + g_down['lat']) / 2
lng_ew = (g_up['lng'] + g_down['lng']) / 2
A_TARGET = get_catchment_area(lat_ew, lng_ew, label='EW')

df_up = df[df['station_id'] == STATION_UP][['date', 'discharge_m3s', 'water_level_cm']].rename(
    columns={'discharge_m3s': 'Q_up', 'water_level_cm': 'level_up'})
df_down = df[df['station_id'] == STATION_DOWN][['date', 'discharge_m3s', 'water_level_cm']].rename(
    columns={'discharge_m3s': 'Q_down', 'water_level_cm': 'level_down'})
df_ew = df_up.merge(df_down, on='date', how='inner').dropna()
df_ew = df_ew[(df_ew['Q_up'] > 0) & (df_ew['Q_down'] > 0)].copy()
df_ew['Q_ew'] = interpolate_q_to_location(
    Q_up=df_ew['Q_up'].values, Q_down=df_ew['Q_down'].values,
    A_up=A_UP, A_down=A_DOWN, A_target=A_TARGET, method='daily_n')
df_q = pd.DataFrame({'station_id': 'EW', 'date': df_ew['date'], 'discharge_m3s': df_ew['Q_ew']})
avg_year_q = average_sorted_year(df_q, 'EW')
Q_sorted = avg_year_q['mean'].values
avg_year_lvl = average_sorted_year(df, STATION_DOWN, column='water_level_cm')
level_sorted = avg_year_lvl['mean'].values / 100.0
level_avg = (df[df['station_id'] == STATION_DOWN]['water_level_cm'].dropna() / 100.0).mean()
H_STAGE = 6.0
H_gross = np.maximum(H_STAGE + (level_avg - level_sorted), 0.0)
Q_ENV = environmental_flow(Q_sorted, method='Q90')

TURBINE_KEYS = ['kaplan', 'propeller', 'crossflow']  # discrete index 0..2

def manning_loss(Q, b, h, L, n_manning=0.013):
    A = b*h; P = b+2*h; R = A/P; v = Q/A
    return (n_manning * v / R**(2/3))**2 * L

def build_loss_fns(n_turbines):
    return [
        lambda Q: trash_rack_loss(Q, A_rack=30.0, bar_width=0.012, bar_spacing=0.05),
        lambda Q: manning_loss(Q, b=10.0, h=2.5, L=30.0, n_manning=0.013),
        lambda Q: minor_loss(Q, A=25.0, xi=0.10),
        lambda Q, A=n_turbines*3.5: minor_loss(Q, A=A, xi=0.10),
        lambda Q, A=n_turbines*6.0: minor_loss(Q, A=A, xi=0.25),
    ]

print(f'Q_sorted: {Q_sorted.min():.1f}..{Q_sorted.max():.1f} m3/s, Q_env={Q_ENV:.1f}')

Q_sorted: 34.7..441.1 m3/s, Q_env=58.7


## Krok 2: Definicja problemu (pymoo)

`ElementwiseProblem` ocenia jedno rozwiązanie naraz. Tu kosztowne (`compute_power` to wiele
operacji) — z 1000-iteracyjnym GA ~50000 wywołań, więc ważne żeby `_evaluate` było szybkie
(stąd parametry domyślne, brak grafiki w pętli).

Zmienne decyzyjne **mieszane**:
- $x_0$ = install_pct (continuous, 10–60)
- $x_1$ = n_turbines (continuous → round do 1..4)
- $x_2$ = turbine_idx (continuous → round do 0..2)

`pymoo` traktuje wszystko jako continuous i zaokrąglamy w `_evaluate` — to standardowa
technika dla mieszanych GA.

In [3]:
ENERGY_PRICE_EUR = 106.5
OM_FRACTION = 0.025
DISCOUNT_RATE = 0.06
LIFETIME = 40

def evaluate_design(install_pct, n_turbines, turbine_key):
    """Ocen pojedyncze rozwiazanie. Zwraca (NPV, LCOE, days_op) lub (None) gdy niewykonalne."""
    install_day = max(1, min(int(install_pct / 100 * len(Q_sorted)), len(Q_sorted) - 1))
    Q_total_design = float(Q_sorted[install_day - 1])
    Q_design_per = Q_total_design / n_turbines
    H_design = float(H_gross[install_day - 1])
    if turbine_key not in TURBINE_CATALOG:
        return None
    turbine = TURBINE_CATALOG[turbine_key]
    if not (turbine.H_range[0] <= H_design <= turbine.H_range[1]):
        return None
    loss_fns = build_loss_fns(n_turbines)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        try:
            result = compute_power(
                Q_sorted=Q_sorted, H_gross=H_gross,
                Q_design=Q_design_per, n_turbines=n_turbines,
                turbine_type=turbine, loss_fns=loss_fns,
                H_design=H_design, Q_env=Q_ENV,
            )
            E_MWh = annual_energy(result)
            if E_MWh <= 0:
                return None
            P_rated_per = 998 * 9.81 * Q_design_per * H_design * turbine.eta_peak / 1000.0
            inv = total_investment(
                P_kW=P_rated_per, H=H_design, n_turbines=n_turbines,
                turbine_type=turbine_key, plant_type='run_of_river_small',
            )
            econ = economic_analysis(
                energy_mwh=E_MWh, investment_eur=inv['total_eur'],
                energy_price_eur_mwh=ENERGY_PRICE_EUR, om_fraction=OM_FRACTION,
                discount_rate=DISCOUNT_RATE, lifetime_years=LIFETIME,
            )
        except Exception:
            return None
    return econ['npv_eur'] / 1e6, econ['lcoe_eur_mwh'], operating_hours(result)

# Sanity check
print('Test:', evaluate_design(install_pct=30, n_turbines=2, turbine_key='kaplan'))

Test: (np.float64(11.069642756593481), np.float64(45.66689272707926), 265)


In [4]:
if PYMOO_OK:
    class EWDesignProblem(ElementwiseProblem):
        """Multi-obj: max NPV (= -minimize), min LCOE; constraint: days_op > 100."""

        def __init__(self):
            super().__init__(
                n_var=3,
                n_obj=2,           # NPV, LCOE
                n_constr=1,        # days_op >= 100
                xl=np.array([10.0, 1.0, 0.0]),     # install_pct, n_turbines, turbine_idx
                xu=np.array([60.0, 4.0, len(TURBINE_KEYS) - 0.001]),
            )

        def _evaluate(self, x, out, *args, **kwargs):
            install_pct = float(x[0])
            n_turbines = max(1, min(int(round(x[1])), 4))
            turbine_idx = max(0, min(int(x[2]), len(TURBINE_KEYS) - 1))
            turbine_key = TURBINE_KEYS[turbine_idx]

            r = evaluate_design(install_pct, n_turbines, turbine_key)
            if r is None:
                # niewykonalne → bardzo zle wartosci
                out['F'] = [1e9, 1e9]
                out['G'] = [365]  # naruszone
                return
            npv_mln, lcoe, days_op = r
            # F: minimize obu, wiec NPV transformujemy: minimize(-NPV) -> max NPV
            out['F'] = [-npv_mln, lcoe]
            out['G'] = [100 - days_op]  # ≤ 0 = constraint satisfied
    print('Klasa EWDesignProblem zdefiniowana.')
else:
    print('pymoo niedostepne — pomijam.')

Klasa EWDesignProblem zdefiniowana.


## Krok 3: Uruchom NSGA-II

Algorytm NSGA-II (Non-dominated Sorting Genetic Algorithm) to standardowy wybór dla
problemów wielokryterialnych. Wykonuje on **selekcję turniejową** opartą na pojeciu
**dominacji Pareto** + **crowding distance** dla zachowania różnorodności na froncie.

Parametry: populacja 100, 50 generacji = ~5000 wywołań `compute_power`.

In [5]:
if PYMOO_OK:
    problem = EWDesignProblem()
    algorithm = NSGA2(
        pop_size=100,
        sampling=FloatRandomSampling(),
        crossover=SBX(prob=0.9, eta=15),
        mutation=PM(eta=20),
        eliminate_duplicates=True,
    )
    print('Uruchamiam NSGA-II (50 generacji × 100 osobnikow = ~5000 ewaluacji)...')
    res = minimize(problem, algorithm,
                    ('n_gen', 50),
                    seed=42, verbose=False)
    print(f'Gotowe. Front Pareto: {len(res.F)} rozwiazan.')
else:
    res = None
    print('Pominieto run (brak pymoo).')

Uruchamiam NSGA-II (50 generacji × 100 osobnikow = ~5000 ewaluacji)...


Gotowe. Front Pareto: 100 rozwiazan.


## Krok 4: Front Pareto

In [6]:
if PYMOO_OK and res is not None and len(res.F) > 0:
    # Odzyskaj rzeczywiste wartosci z transformowanych
    npv_vals = -res.F[:, 0]
    lcoe_vals = res.F[:, 1]
    # Dekoduj zmienne decyzyjne
    decisions = []
    for x in res.X:
        ip = float(x[0])
        nt = max(1, min(int(round(x[1])), 4))
        ti = max(0, min(int(x[2]), len(TURBINE_KEYS) - 1))
        decisions.append({
            'install_pct': round(ip, 1),
            'n_turbines': nt,
            'turbine': TURBINE_KEYS[ti],
        })
    df_front = pd.DataFrame(decisions)
    df_front['NPV [mln EUR]'] = np.round(npv_vals, 2)
    df_front['LCOE [EUR/MWh]'] = np.round(lcoe_vals, 1)
    df_front = df_front.sort_values('LCOE [EUR/MWh]').reset_index(drop=True)

    print(f'Front Pareto ({len(df_front)} rozwiazan, posortowane wg LCOE):')
    print(df_front.to_string(index=False))

    # Wykres frontu Pareto
    fig = go.Figure()
    color_map = {'kaplan': '#1976D2', 'propeller': '#388E3C', 'crossflow': '#F57C00'}
    for turb in TURBINE_KEYS:
        mask = df_front['turbine'] == turb
        if mask.any():
            fig.add_trace(go.Scatter(
                x=df_front.loc[mask, 'LCOE [EUR/MWh]'],
                y=df_front.loc[mask, 'NPV [mln EUR]'],
                mode='markers+text', name=turb,
                text=[f'{r["n_turbines"]}×@{r["install_pct"]:.0f}%'
                       for _, r in df_front[mask].iterrows()],
                textposition='top center',
                marker=dict(size=12, color=color_map[turb], line=dict(width=1, color='black')),
            ))
    fig.update_layout(
        title='Front Pareto: NPV vs LCOE (etykiety: n_turbines × install_pct)',
        xaxis_title='LCOE [EUR/MWh]  (mniejsze = lepiej)',
        yaxis_title='NPV [mln EUR]  (wieksze = lepiej)',
        height=550,
    )
    fig.show()
else:
    print('Brak wyniku — sprawdz czy pymoo zainstalowane.')

Front Pareto (100 rozwiazan, posortowane wg LCOE):
 install_pct  n_turbines   turbine  NPV [mln EUR]  LCOE [EUR/MWh]
        60.0           2 crossflow          18.26            20.2
        59.9           2 crossflow          18.26            20.2
        59.8           3 crossflow          21.42            20.8
        59.8           3 crossflow          21.42            20.8
        59.6           3 crossflow          21.43            20.8
        58.7           3 crossflow          21.46            20.9
        58.6           3 crossflow          21.47            20.9
        59.1           3 crossflow          21.45            20.9
        59.3           3 crossflow          21.44            20.9
        57.9           3 crossflow          21.48            21.0
        57.6           3 crossflow          21.48            21.0
        57.6           3 crossflow          21.48            21.0
        58.2           3 crossflow          21.47            21.0
        57.5           3 

## Krok 5: Wybor punktu — kompromis (knee point)

Front Pareto **nie wskazuje** jednego "najlepszego" rozwiązania — wszystkie są optymalne
w sensie Pareto. Wybór zależy od **preferencji** inwestora:

- **Skraj NPV** (góra): maksymalny zysk całkowity, ale wyższy koszt jednej MWh
- **Skraj LCOE** (lewo): najniższy koszt produkcji, ale mniejszy zysk całkowity
- **Punkt kompromisowy** (środek, *knee*): zwykle wybor inżynierski — gdzie krzywa
  najbardziej się zagina

Standardowa heurystyka *knee point*: punkt na froncie najdalej od prostej łączącej skrajne.

In [7]:
if PYMOO_OK and res is not None and len(res.F) >= 3:
    # Normalizuj NPV i LCOE do [0, 1]
    npv_n = (npv_vals - npv_vals.min()) / max(npv_vals.max() - npv_vals.min(), 1e-9)
    lcoe_n = (lcoe_vals - lcoe_vals.min()) / max(lcoe_vals.max() - lcoe_vals.min(), 1e-9)
    # Skraj 1: max NPV (npv_n=1, lcoe_n=?), Skraj 2: min LCOE (lcoe_n=0)
    # Linia od (0, 1) do (1, 0) na (LCOE_n, -NPV_n) — chcemy najdalej od niej
    # Distance from line ax+by+c=0: |a*x + b*y + c| / sqrt(a²+b²)
    # Line through (0,1) and (1,0): x + y - 1 = 0
    distances = np.abs(lcoe_n + (1 - npv_n) - 1) / np.sqrt(2)
    knee_idx = int(np.argmax(distances))
    knee = {
        'NPV': npv_vals[knee_idx],
        'LCOE': lcoe_vals[knee_idx],
        'install_pct': float(res.X[knee_idx, 0]),
        'n_turbines': max(1, min(int(round(res.X[knee_idx, 1])), 4)),
        'turbine': TURBINE_KEYS[max(0, min(int(res.X[knee_idx, 2]), len(TURBINE_KEYS) - 1))],
    }
    print('Knee point (rekomendowany kompromis):')
    for k, v in knee.items():
        print(f'  {k}: {v}')
else:
    print('Za malo punktow na froncie zeby liczyc knee.')

Knee point (rekomendowany kompromis):
  NPV: 21.42107437339757
  LCOE: 20.815210893009436
  install_pct: 59.79361218076492
  n_turbines: 3
  turbine: crossflow


## Podsumowanie

1. **Optymalizacja wielokryterialna** nie zwraca jednego optymalnego punktu — zwraca
   **front Pareto** rozwiązań niezdominowanych.
2. **NSGA-II** (Non-dominated Sorting GA II) to klasyczny algorytm dla 2-3 celów.
3. **Mieszane zmienne** (continuous + discrete) traktujemy jako wszystkie continuous
   z zaokrąglaniem w `_evaluate`.
4. **Wybór z frontu** wymaga *artykulacji preferencji* — knee point to powszechna heurystyka,
   ale można też zważyć cele (`weighted sum`) lub użyć metod ELECTRE/AHP.

### Ograniczenia tego notebooka

- GA jest **stochastyczny** — dwa runy mogą dać nieco różne fronty. Używamy `seed=42`
  dla powtarzalności.
- Funkcja celu `compute_power` jest **gładka** (głównie analityczne wzory + interpolacje),
  więc gradient-based methods (NLP) też by zadziałały i byłyby szybsze. GA jednak
  lepiej obsługuje zmienne dyskretne.
- Cykl 50 gen × 100 osob może zająć kilka minut. Dla szybszej analizy zmniejsz pop_size
  lub liczbę generacji.

**Możliwe rozszerzenia** (eksperymenty studentów):

- Dodaj 3-ci cel: max capacity factor
- Dodaj 4-ą zmienną: $D_{rurociag}$ (wpływa na straty)
- Dodaj ograniczenie cavitation OK z `src.turbine.suction_head_max`
- Porównaj NSGA-II z NSGA-III dla > 3 celów